In [38]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(model = 'gpt-4o-mini')

In [39]:
from typing import Literal
from pydantic import BaseModel, Field


class SentimentSchema(BaseModel):
    sentiment : Literal["positive", "negative"] = Field(description="Sentiment of the review")

In [40]:
structured_model = model.with_structured_output(SentimentSchema)



In [41]:
prompt = "What is the sentiment of the following review- The software too good"

structured_model.invoke(prompt).sentiment


'positive'

In [44]:
class ReviewState(TypedDict):

    review: str
    sentiment : Literal["positive", "negative"]
    diagnosis: dict
    response: str
    

In [45]:
class DaignosisSchema(BaseModel):
    issue_type : Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description = 'The category of issue mentioned in the review' )
    tone : Literal['angry', 'frustrated', 'disapprointed', 'calm'] = Field(description='The emotional tone expressed by the user')
    urgency : Literal['high', 'medium', 'low'] = Field(description='How urgent or critical the issues appears to be')

structured_model2 = model.with_structured_output(DaignosisSchema)

In [48]:
from tkinter import N


def find_sentiment(state: ReviewState):

    prompt = f"for the following review find out the sentiment \n {state["review"]}"


    sentiment = structured_model.invoke(prompt).sentiment

    return {
        'sentiment' : sentiment
    }


def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'


def positive_response(state: ReviewState):

    prompt = f""" Write a warm thank you message in reponse to this review:
    \n\n {state['review']} \n 

    Also, kindly ask the user to leave feedback on our websites. """
    
    response =  model.invoke(prompt).content

    return {
        "response" : response
    }


def run_diagonis(state: ReviewState):
    prompt = f"""Diagonose this negative review \n\n {state['review']} \n  Return issue_type, tome and urgency."""

    response = structured_model2.invoke(prompt) 
    return {
        "diagnosis" : response.model_dump()
    }

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']
    
    prompt = f""" You are a support assistant.

     The user had a {diagnosis['issue_type']} issue, sounded {diagnosis['tone']} and marked urgency as {diagnosis['urgency']}

     Write a empathy, helpful resolution message """

    response = model.invoke(prompt).content
    return {
        'response' : response
    }




In [50]:
graph = StateGraph(ReviewState)


graph.add_node('find_sentiment', find_sentiment)

graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagonis)
graph.add_node('negative_response', negative_response)


graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')

graph.add_edge('negative_response', END)

workflow = graph.compile()

In [51]:
intial_state = {
    'review' : 'The product on website is not loading fix it as soon as possible'
}

workflow.invoke(intial_state)

{'review': 'The product on website is not loading fix it as soon as possible',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Performance',
  'tone': 'frustrated',
  'urgency': 'high'},
 'response': "Subject: We're Here to Help with Your Performance Issues\n\nDear [User's Name],\n\nI hope this message finds you well. I want to first acknowledge your frustration regarding the performance issue you've been experiencing. I can only imagine how disruptive this has been for you, especially with urgency marked high. \n\nWe are committed to ensuring you have a smooth experience, and I’m here to assist you in resolving this as quickly as possible. To help us pinpoint the problem, could you please provide a bit more detail about the specific performance issues you're encountering? Additionally, any information about your system setup or recent changes can also be very helpful.\n\nIn the meantime, here are a few troubleshooting steps that might alleviate the issue:\n\n1. **Restart Your 